# C6-pytorch — Practice p15 — Solution


The two hidden scores are $\text{temperature}-45$ and
$20-\text{humidity}$.  After thresholding, their values are bits.
An OR fires when their sum is at least one, represented by the readout
score $h_1+h_2-0.5$ (the pinned OR bias) before the final inclusive gate.


In [ ]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)

class DenseLayer(nn.Module):
    """Session 2's pinned dense layer: weight (out, in), bias (out,)."""

    def __init__(self, weight, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.as_tensor(weight), requires_grad=False)
        self.bias = nn.Parameter(torch.as_tensor(bias), requires_grad=False)

    def forward(self, x):
        return x @ self.weight.T + self.bias

class ThresholdGate(nn.Module):
    """Session 2's gate: 1 where x >= 0, else 0; owns no parameters."""

    def forward(self, x):
        return (x >= 0).to(x.dtype)


readings = torch.tensor([[50.0, 50.0], [30.0, 10.0], [50.0, 15.0],
                         [30.0, 50.0], [45.0, 20.0], [44.9, 20.1]])
expected = torch.tensor([1.0, 1.0, 1.0, 0.0, 1.0, 0.0])

W1 = torch.tensor([[1.0, 0.0], [0.0, -1.0]])
b1 = torch.tensor([-45.0, 20.0])
W2 = torch.tensor([[1.0, 1.0]])
b2 = torch.tensor([-0.5])   # OR: -0.5, the course's pinned half-integer bias

class Alarm(nn.Module):
    def __init__(self):
        super().__init__()
        self.conditions = DenseLayer(W1, b1)
        self.gate = ThresholdGate()
        self.readout = DenseLayer(W2, b2)

    def forward(self, x):
        return self.gate(self.readout(self.gate(self.conditions(x))))


alarm = Alarm()
fired = alarm(readings).ravel()
gap = float((fired - expected).abs().max())

W1, b1, W2, b2, fired, gap


These weights are a direct encoding of the two policy inequalities and
Boolean OR; every coefficient and bias comes from moving a rule to a
“score $\ge 0$” form.  No data fitting is needed: deployment consists
only of evaluating those fixed scores and gates on each reading.


### Answer check


In [ ]:
assert torch.equal(fired, expected)
assert fired[4].item() == 1.0
assert fired[5].item() == 0.0
assert gap == 0.0
assert all(not p.requires_grad for p in alarm.parameters())
